In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (scatole, fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>
#include <random>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Scatola { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1


// AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
// Stanza = cubo [-1, 1]^3 (20 m di lato), z verticale (z = -1 pavimento). Le 6 pareti sono ostacoli (PARETI).
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

// Le 6 pareti della stanza sono ostacoli; i muri interni con i fori li crea genera_mappa (sotto)
const double SP = 0.04;                                  // spessore delle pareti della stanza
const std::vector<Scatola> PARETI = {                    // le 6 pareti della stanza (pavimento e soffitto compresi)
    { -1.0, -1.0 + SP, -1.0, 1.0, -1.0, 1.0 }, { 1.0 - SP, 1.0, -1.0, 1.0, -1.0, 1.0 },
    { -1.0, 1.0, -1.0, -1.0 + SP, -1.0, 1.0 }, { -1.0, 1.0, 1.0 - SP, 1.0, -1.0, 1.0 },
    { -1.0, 1.0, -1.0, 1.0, -1.0, -1.0 + SP }, { -1.0, 1.0, -1.0, 1.0, 1.0 - SP, 1.0 },
};
// Mappe casuali: K muri perpendicolari a x, ognuno con M fori quadrati di lato casuale.
// A nel primo scomparto, B nell'ultimo. Stesso seed = stessa mappa.
const double SPESSORE = 0.10;                            // spessore dei muri interni
const double X_PRIMO = -0.6, X_ULTIMO = 0.6;             // centro (in x) del primo e dell'ultimo muro
inline Punto A, B;                                       // partenza e arrivo: li sceglie genera_mappa

inline std::vector<Scatola> SCATOLE;                     // ostacoli della mappa attuale (pareti + muri)
inline std::vector<Scatola> FORI;                        // zone in cui l'ostacolo viene tolto

inline void genera_mappa(int K, int M, double lato_min, double lato_max, unsigned seed)
{
    std::mt19937 rng(seed);                                                     // generatore di numeri casuali
    auto caso = [&](double a, double b) { return std::uniform_real_distribution<double>(a, b)(rng); };   // numero casuale in [a, b]
    SCATOLE = PARETI;
    FORI.clear();
    for (int w = 0; w < K; ++w) {
        const double xc = (K == 1) ? 0.0 : X_PRIMO + (X_ULTIMO - X_PRIMO) * w / (K - 1);   // centro del muro w: muri equidistanti
        const double x0 = xc - SPESSORE / 2, x1 = xc + SPESSORE / 2;
        SCATOLE.push_back({ x0, x1, -1.0, 1.0, -1.0, 1.0 });                    // il muro chiude tutta la sezione della stanza
        for (int f = 0; f < M; ++f) {
            const double lato = caso(lato_min, lato_max);                       // lato del foro
            const double lim = 1.0 - SP - MARGINE - lato / 2;                   // il foro non tocca le pareti della stanza
            const double yc = caso(-lim, lim), zc = caso(-lim, lim);            // centro del foro sul muro
            FORI.push_back({ x0, x1, yc - lato / 2, yc + lato / 2, zc - lato / 2, zc + lato / 2 });
        }
    }
    const double xa = X_PRIMO - SPESSORE / 2, xb = X_ULTIMO + SPESSORE / 2;     // faccia sinistra del primo muro, destra dell'ultimo
    A = { caso(-1.0 + SP + 0.1, xa - 0.1), caso(-0.8, 0.8), caso(-0.8, 0.8) };  // primo scomparto, a piu' di 0.1 da ogni ostacolo
    B = { caso(xb + 0.1, 1.0 - SP - 0.1), caso(-0.8, 0.8), caso(-0.8, 0.8) };   // ultimo scomparto
}
// Livelli di difficolta': piu' muri, meno fori per muro, fori piu' piccoli
struct Livello { int K, M; double lato_min, lato_max; };    // K muri, M fori per muro, lato dei fori in [lato_min, lato_max]
const std::vector<Livello> LIVELLI = {
    { 2, 3, 0.30, 0.50 },                                   // L1: facile
    { 3, 2, 0.25, 0.45 },                                   // L2
    { 4, 2, 0.20, 0.40 },                                   // L3
    { 5, 2, 0.15, 0.35 },                                   // L4
    { 6, 2, 0.15, 0.30 },                                   // L5: difficile (2 fori per muro: uno si puo' chiudere)
};
// Ripianificazione: quanti fori aperti ha il muro del foro f (i fori di uno stesso muro hanno la stessa x0)
inline int fori_aperti_nel_muro(const Scatola& f)
{
    int n = 0;
    for (const auto& g : FORI) if (g.x0 == f.x0) ++n;
    return n;
}
// =============================================================================

// Vero se il punto (x, y, z) e' dentro la scatola s (bordi compresi)
inline bool dentro(const Scatola& s, double x, double y, double z)
{
    return s.x0 <= x && x <= s.x1 && s.y0 <= y && y <= s.y1 && s.z0 <= z && z <= s.z1;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& s : SCATOLE) if (dentro(s, x, y, z)) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                     // un nodo alla volta
    const int j = q % N, i = (q / N) % N, k = q / (N * N);
    mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;
    }
    // Margine: diventa ostacolo ogni nodo entro MARGINE da un ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio sfera intorno all'ostacolo
    std::vector<unsigned char> margin_mask = mask;
    for (int q = 0; q < n; ++q) {
        if (mask[q]) continue;                                   
    const int k = q / (N * N), i = (q / N) % N, j = q % N;
    const bool bordo = (j > 0 && mask[q - 1]) || (j < N - 1 && mask[q + 1]) ||          // che toccano un nodo libero
                       (i > 0 && mask[q - N]) || (i < N - 1 && mask[q + N]) ||
                       (k > 0 && mask[q - N * N]) || (k < N - 1 && mask[q + N * N]);
    if (!bordo) continue;
    for (int dk = -r; dk <= r; ++dk)                          // tutti i nodi entro la sferadi raggio MARGINE
        for (int di = -r; di <= r; ++di)
            for (int dj = -r; dj <= r; ++dj) {
                const int kk = k + dk, ii = i + di, jj = j + dj;
                if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                    margin_mask[(kk * N + ii) * N + jj] = 0;
            }
}
return margin_mask;
}
// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}
// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FMM (CPU) sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile block_fim3d.cuh
#pragma once           // il file viene incluso una sola volta
// Block FIM 3D (Alg. 7) sulla GPU: blocchi 8^3 in shared memory, uscita anticipata, maschera degli ostacoli
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "fmm3d.h"         // INF e solve_quadratic3 (lo stesso risolutore locale del FMM)

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out)
{
    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    __shared__ bool s_Cn[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q;
        if (fixed) {q = p;}
        else{
        // tx, ty, tz: 0,...,7; +1: nella shared memory c'è l'halo
        q = solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        }    
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break;   // tutti hanno scritto; se nessun nodo e' cambiato, esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cn del nodo: vero se non è cambiato nell'ultima iterazione
    s_Cn[tid] = conv;
    __syncthreads();
    for (int reduct = BLOCK * BLOCK * BLOCK / 2; reduct > 0; reduct /= 2) {     // 256, ..., 4, 2, 1
        if (tid < reduct) s_Cn[tid] = s_Cn[tid] && s_Cn[tid + reduct];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_Cn[0];
}
// Step1: ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* active_list, float* u, const float* F,
                                    int N, int nbx, float h, const unsigned char* mask, int niter,
                                    float eps, unsigned char* Cb_step1)
{
    // blockIdx è il numero del blocco CUDA nel lancio in corso
    process_block(active_list[blockIdx.x], niter, u, F, N, nbx, h, mask, eps, Cb_step1);
    // (es. N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (es. N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Step 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* is_candidate,
                                        float* u, const float* F, int N, int nbx, float h,
                                        const unsigned char* mask, float eps, unsigned char* Cb)
{
    // scarta blocchi CUDA in eccesso
    if (blockIdx.x >= *cand_count) return;
    int b = cand_list[blockIdx.x];
    // blocchi candidati: niter = 1 -> con la nuova informazione dal blocco convergente, questo vicino cambia?
    process_block(b, 1, u, F, N, nbx, h, mask, eps, Cb);
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) is_candidate[b] = 0;   // pronto per il giro successivo
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // bytes: dimensione totale griglia 3D
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb_step1, *d_in_list, *d_mask; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_mask, N*N*N));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), N*N*N, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_active_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, d_u, d_F, N, nbx, h, d_mask, niter, eps, d_Cb_step1);    // si lanciano n_active blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_candidate_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_is_candidate,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb_step1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}


In [ ]:
%%writefile main.cu
// FM2 con il FMM e con il Block FIM su mappe casuali, con 3 ripianificazioni per mappa (un foro si chiude davanti al drone)
#include <cstdio>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include <string>          // nomi dei file delle mappe
#include <cuda_runtime.h>  // prima di fmm3d.h: definisce __host__ __device__
#include "fmm3d.h"         // Fast Marching (CPU)
#include "ambiente.h"      // stanza, ostacoli, maschera, lentezza del FM2
#include "block_fim3d.cuh" // Block FIM (GPU)

// FM2 completo (passata 1 + passata 2) sulla mappa attuale, con il FMM (CPU) o con il Block FIM (GPU).
// Restituisce u (tempo di arrivo da B), la F del FM2 e il tempo di calcolo delle due passate [ms].
void fm2(bool con_block_fim, int Nk, const std::vector<unsigned char>& mask, float eps, int niter,
         std::vector<float>& u, std::vector<float>& Fk, double& t_ms, int& giri)
{
    const float hk = 2.0f / (Nk - 1);
    const int n = Nk * Nk * Nk;
    // prima passata: numero di nodi liberi che toccano un ostacolo
    const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
    // seconda passata: percorso minimo per B          
    const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          
    std::vector<float> d(n, INF);
    for (int s : bordo) d[s] = 0.0f;
    const std::vector<float> uno(n, 1.0f);
    u.assign(n, INF);
    for (int s : srcs) u[s] = 0.0f;
    giri = 0;
    if (con_block_fim) {                                     // GPU: tempo del solo ciclo dei giri (come nelle tabelle)
        float ms1 = 0.0f, ms2 = 0.0f;
        // calcolo la distanza dei nodi liberi dagli ostacoli
        block_fim(d, uno, Nk, hk, bordo, mask, eps, niter, &ms1);
        // definisco la lentezza
        Fk = lentezza_fm2(d);
        // tempo di arrivo in B
        giri = block_fim(u, Fk, Nk, hk, srcs, mask, eps, niter, &ms2);
        // tempo impiegato dal FM2 con Block FIM
        t_ms = ms1 + ms2;
    } else {                                                 // CPU: le due chiamate al FMM
        auto t0 = std::chrono::high_resolution_clock::now();
        fmm(d, uno, Nk, hk, bordo, mask);
        Fk = lentezza_fm2(d);
        fmm(u, Fk, Nk, hk, srcs, mask);
        auto t1 = std::chrono::high_resolution_clock::now();
        t_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
    }
}

// Scrive i file di una mappa per Python: u, maschera, F e ambiente (ostacoli e fori della mappa attuale)
void scrivi_mappa(int Nk, const std::string& nome, const std::vector<float>& u, const std::vector<unsigned char>& mask, const std::vector<float>& Fk)
{
    const std::string t = std::to_string(Nk) + "_" + nome;
    std::ofstream out("fim_" + t + ".bin", std::ios::binary);
    out.write(reinterpret_cast<const char*>(u.data()), u.size() * sizeof(float));
    std::ofstream om("mask_" + t + ".bin", std::ios::binary);
    om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
    std::ofstream of("F_" + t + ".bin", std::ios::binary);
    of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
    std::ofstream amb("ambiente_" + t + ".txt");
    amb.precision(15);
    amb << "MARGINE " << MARGINE << "\n";
    amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
    amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
    for (const auto& s : SCATOLE) amb << "SCATOLA " << s.x0 << " " << s.x1 << " " << s.y0 << " " << s.y1 << " " << s.z0 << " " << s.z1 << "\n";
    for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
}

// RIPIANIFICAZIONE
const int RIPIANIFICAZIONI = 3;                              // quante volte si chiude un foro durante il volo
const double ANTICIPO = 0.15;                                // il foro si chiude quando il drone e' a ~0.15 (1.5 m) di percorso da esso

// Coordinate (x, y, z) del nodo q
inline Punto punto(int N, int q)
{
    const double h = 2.0 / (N - 1);
    return { -1.0 + (q % N) * h, -1.0 + ((q / N) % N) * h, -1.0 + (q / (N * N)) * h };
}

// Percorso dal nodo p fino a B: a ogni passo il vicino libero (26 vicini) con la u piu' piccola
std::vector<int> discesa(const std::vector<float>& u, const std::vector<unsigned char>& mask, int N, int p)
{
    std::vector<int> perc = { p };
    const int pB = nodo(N, B.x, B.y, B.z);
    while (p != pB) {
        const int k = p / (N * N), i = (p / N) % N, j = p % N;
        int migliore = p;
        for (int dk = -1; dk <= 1; ++dk)
            for (int di = -1; di <= 1; ++di)
                for (int dj = -1; dj <= 1; ++dj) {
                    const int kk = k + dk, ii = i + di, jj = j + dj;
                    if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                    const int q = (kk * N + ii) * N + jj;
                    if (mask[q] && u[q] < u[migliore]) migliore = q;
                }
        if (migliore == p) break;                            // nessun vicino piu' basso: ci si ferma
        p = migliore;
        perc.push_back(p);
    }
    return perc;
}

// Il primo foro del percorso che si puo' chiudere (il suo muro ha almeno un altro foro aperto).
// Restituisce la sua posizione in FORI (-1 se non ce n'e') e in k il punto del percorso in cui e' il drone quando si chiude.
int prossimo_foro(const std::vector<int>& perc, int N, int& k)
{
    const int prima = (int)std::lround(ANTICIPO / (2.0 / (N - 1)));     // ANTICIPO in nodi
    for (size_t s = 0; s < perc.size(); ++s) {
        const Punto p = punto(N, perc[s]);
        for (size_t f = 0; f < FORI.size(); ++f)
            if (dentro(FORI[f], p.x, p.y, p.z) && fori_aperti_nel_muro(FORI[f]) >= 2) {
                k = std::max(0, (int)s - prima);
                return (int)f;
            }
    }
    return -1;
}

int main()
{
    // Informazioni sulla GPU usata
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 8;                                     // iterazioni locali massime (uscita anticipata): la migliore con gli ostacoli (prove 12 / 24)

    struct Prova { int N, S; };                              // griglia e numero di mappe per livello
    const std::vector<Prova> PROVE = { { 129, 10 }, { 193, 5 }, { 257, 3 } };   // come il test senza ripianificazione (3 ripianificazioni per mappa)

    for (const auto& p : PROVE) {
        const int Nk = p.N;
        std::ofstream csv("ripianificazione_" + std::to_string(Nk) + ".csv");    // una riga per ripianificazione
        csv << "livello,seed,rip,muro_x,drone_x,drone_y,drone_z,u_drone,giri,t_fmm,t_bfim,speedup,max_diff\n";
        printf("\nN = %d\n  liv  seed  rip   muro x   u(drone)   giri   max|FMM-BFIM|   t FMM [ms]   t BFIM [ms]   speedup\n", Nk);
        for (int l = 0; l < (int)LIVELLI.size(); ++l) {
            const Livello& L = LIVELLI[l];
            for (int s = 0; s < p.S; ++s) {
                const unsigned seed = 1000 * l + s;
                genera_mappa(L.K, L.M, L.lato_min, L.lato_max, seed);
                std::vector<unsigned char> mask = costruisci_maschera(Nk);
                std::vector<float> u_fmm, u_bfim, F_fmm, F_bfim;
                double t;
                int g;
                fm2(true, Nk, mask, eps, niter, u_bfim, F_bfim, t, g);           // piano iniziale (non si misura)
                int pos = nodo(Nk, A.x, A.y, A.z);                               // il drone parte da A
                const bool salva = (Nk == 129 && s == 0);                        // prima mappa di ogni livello: file per la gara 3D
                const std::string base = "livello" + std::to_string(l + 1);
                std::ofstream tempi;                                             // tempi di ogni ripianificazione, per Python
                if (salva) { scrivi_mappa(Nk, base + "_stato0", u_bfim, mask, F_bfim); tempi.open("tempi_" + base + ".txt"); }
                for (int r = 1; r <= RIPIANIFICAZIONI; ++r) {
                    const std::vector<int> perc = discesa(u_bfim, mask, Nk, pos);   // percorso attuale del drone
                    int k = 0;
                    const int f = prossimo_foro(perc, Nk, k);
                    if (f < 0) { printf("  L%d  %5u  %3d   nessun foro da chiudere\n", l + 1, seed, r); break; }
                    pos = perc[k];                                               // il drone e' qui quando il foro si chiude
                    const double muro_x = (FORI[f].x0 + FORI[f].x1) / 2;
                    FORI.erase(FORI.begin() + f);                                // il foro si chiude
                    mask = costruisci_maschera(Nk);
                    double t_fmm = 0.0, t_bfim = 1e30;
                    int giri = 0;
                    fm2(false, Nk, mask, eps, niter, u_fmm, F_fmm, t_fmm, g);    // ripianificazione con il FMM: 1 misura
                    for (int m = 0; m < 3; ++m) {                                // con il Block FIM: il minimo di 3
                        fm2(true, Nk, mask, eps, niter, u_bfim, F_bfim, t, giri);
                        t_bfim = std::min(t_bfim, t);
                    }
                    float diff = 0.0f;
                    for (size_t q = 0; q < u_fmm.size(); ++q) diff = fmaxf(diff, fabsf(u_fmm[q] - u_bfim[q]));
                    const Punto d = punto(Nk, pos);
                    const float ud = mask[pos] ? u_bfim[pos] : -1.0f;            // tempo che resta al drone (-1: e' nel margine del foro chiuso)
                    csv << l + 1 << "," << seed << "," << r << "," << muro_x << "," << d.x << "," << d.y << "," << d.z << "," << ud << ","
                        << giri << "," << t_fmm << "," << t_bfim << "," << t_fmm / t_bfim << "," << diff << "\n";
                    printf("  L%d  %5u  %3d   %6.2f   %8.4f   %4d   %13.2e   %10.1f   %11.2f   %7.1f\n",
                           l + 1, seed, r, muro_x, ud, giri, diff, t_fmm, t_bfim, t_fmm / t_bfim);
                    if (salva) {                                                 // la mappa dopo la chiusura r e i tempi per ricalcolarla
                        scrivi_mappa(Nk, base + "_stato" + std::to_string(r), u_bfim, mask, F_bfim);
                        tempi << r << " " << t_fmm << " " << t_bfim << "\n";
                    }
                }
            }
        }
    }
    return 0;
}


In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 main.cu -o fim3d && ./fim3d

In [ ]:
# GARA CON RIPIANIFICAZIONE SU UNA MAPPA CASUALE (usa i file scritti da main.cu a N = 129): scegli il livello e riesegui la cella.
# Due viste affiancate (sinistra Block FIM, destra FMM): i due droni partono insieme da A. Quando un drone e' a ANTICIPO
# (1.5 m di percorso) dal foro in cui sta per passare, quel foro si chiude (fino a 3 volte, gli stessi fori di main.cu).
# A ogni chiusura il drone ricalcola il FM2 per il tempo del suo metodo continuando sul percorso vecchio; se il prossimo
# punto e' nel foro chiuso frena e aspetta. La scia e' colorata con la velocita'; mentre ricalcola il pallino lampeggia.
import csv, os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import RegularGridInterpolator

# 0) PARAMETRI
LIVELLO = 3                                                         # 1 (facile) ... 5 (difficile)
N = 129                                                             # griglia dei file scritti da main.cu
TEMPI_N = 257                                                       # tempi di ricalcolo misurati a questa griglia (stessa mappa, stessi seed)
METRI_PER_UNITA = 10.0                                              # il cubo [-1, 1]^3 e' una stanza di 20 m di lato
V_MAX = 2.0                                                         # velocita' massima del drone [m/s], dove F = 1
SEC_PER_U = METRI_PER_UNITA / V_MAX                                 # 1 unita' di u = 5 s di volo
ANTICIPO = 0.15                                                     # come in main.cu: il foro si chiude a 0.15 (1.5 m) di percorso
MS_FRAME = 60                                                       # il Play mostra un fotogramma ogni 60 ms
BLU, ARANCIO, ROSSO = "#2a78d6", "#eb6834", "#c0392b"
h = 2.0 / (N - 1)
x = -1.0 + h * np.arange(N)

# 1) DATI: le mappe prima e dopo ogni chiusura (u da B, maschera, lentezza F, ostacoli) e i tempi di ricalcolo
def leggi_ambiente(nome):
    amb = {"SCATOLA": [], "FORO": []}
    for riga in open(nome):
        k, *v = riga.split(); v = [float(t) for t in v]
        if k in ("A", "B"): amb[k] = np.array(v)
        elif k == "MARGINE": amb[k] = v[0]
        else: amb[k].append(tuple(v))
    return amb
mappe = []
while os.path.exists(f"fim_{N}_livello{LIVELLO}_stato{len(mappe)}.bin"):
    t = f"{N}_livello{LIVELLO}_stato{len(mappe)}"
    u = np.fromfile(f"fim_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)   # u[k, i, j] = u(z, y, x)
    libero = np.fromfile(f"mask_{t}.bin", dtype=np.uint8).reshape(N, N, N) == 1
    F = np.fromfile(f"F_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)
    mappe.append(dict(u=u, libero=libero, F=F, amb=leggi_ambiente(f"ambiente_{t}.txt"),
                      F_i=RegularGridInterpolator((x, x, x), F), peso=RegularGridInterpolator((x, x, x), libero.astype(float))))
A, B = mappe[0]["amb"]["A"], mappe[0]["amb"]["B"]
chiuso = {j: [f for f in mappe[j - 1]["amb"]["FORO"] if f not in mappe[j]["amb"]["FORO"]][0] for j in range(1, len(mappe))}   # foro chiuso alla chiusura j
seed = 1000 * (LIVELLO - 1)
tempi = {}                                                          # tempo di ricalcolo [s] per chiusura e metodo
if os.path.exists(f"ripianificazione_{TEMPI_N}.csv"):
    for r in csv.DictReader(open(f"ripianificazione_{TEMPI_N}.csv")):
        if int(r["livello"]) == LIVELLO and int(r["seed"]) == seed:
            tempi[int(r["rip"])] = {"Block FIM": float(r["t_bfim"]) / 1000.0, "FMM": float(r["t_fmm"]) / 1000.0, "muro_x": float(r["muro_x"])}
    sorgente = f"misurati a N = {TEMPI_N} sulla stessa mappa"
    x_muri = [round((f[0] + f[1]) / 2, 3) for f in chiuso.values()]
    if [round(tempi[j]["muro_x"], 3) for j in sorted(tempi)][:len(x_muri)] != x_muri:
        print(f"Attenzione: a N = {TEMPI_N} si chiudono fori su muri diversi ({[tempi[j]['muro_x'] for j in sorted(tempi)]}) da N = {N} ({x_muri})")
else:                                                               # niente CSV a TEMPI_N: tempi misurati a N = 129
    for riga in open(f"tempi_livello{LIVELLO}.txt"):
        r_, t_fmm, t_bfim = riga.split()
        tempi[int(r_)] = {"Block FIM": float(t_bfim) / 1000.0, "FMM": float(t_fmm) / 1000.0}
    sorgente = f"misurati a N = {N}"

# 2) PERCORSO da un punto qualsiasi su una mappa: discesa lungo u (26 vicini) + accorciamento
mosse = [(a, b, c) for a in (-1, 0, 1) for b in (-1, 0, 1) for c in (-1, 0, 1) if (a, b, c) != (0, 0, 0)]
nodo = lambda p: tuple(np.round((np.asarray(p)[::-1] + 1.0) / h).astype(int))
coord = lambda n: -1.0 + h * np.array(n[::-1], dtype=float)
def tempo(mp, p, q):                                                # tempo (unita' di u) lungo il segmento p-q: somma di F * ds
    s = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.sum(mp["F_i"]((p + (s[:-1] + s[1:]) / 2 * (q - p))[:, ::-1])) * np.linalg.norm(q - p) / (len(s) - 1)
def si_vede(mp, p, q):                                              # il segmento p-q non attraversa ostacoli
    s = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.all(mp["peso"]((p + s * (q - p))[:, ::-1]) > 0.999)
def percorso(mp, p0):
    u, libero, F = mp["u"], mp["libero"], mp["F"]
    n, nB = nodo(p0), nodo(B)
    if not libero[n] or u[n] >= 1e29:                               # nel margine del foro appena chiuso: si parte dal vicino libero piu' basso
        vic = [(n[0] + a, n[1] + b, n[2] + c) for a, b, c in mosse]
        vic = [w for w in vic if all(0 <= s < N for s in w) and libero[w] and u[w] < 1e29]
        if not vic: raise SystemExit(f"il drone in {np.round(p0, 2)} non raggiunge B")
        n = min(vic, key=lambda w: u[w])
    nodi = [n]
    while n != nB:
        vic = [(n[0] + a, n[1] + b, n[2] + c) for a, b, c in mosse]
        vic = [w for w in vic if all(0 <= s < N for s in w) and libero[w]]
        n = min(vic, key=lambda w: u[w] + 0.5 * (F[n] + F[w]) * h * np.linalg.norm(np.subtract(w, n)))
        nodi.append(n)
    P = [np.asarray(p0, float)] + [coord(w) for w in nodi[1:]]
    tn = np.concatenate([[0], np.cumsum([tempo(mp, P[i], P[i + 1]) for i in range(len(P) - 1)])])
    out, a = [P[0]], 0
    while a < len(P) - 1:
        b = len(P) - 1
        while b > a + 1 and not (si_vede(mp, P[a], P[b]) and tempo(mp, P[a], P[b]) <= tn[b] - tn[a]):
            b -= 1
        out.append(P[b]); a = b
    return np.array(out)
def orario(mp, perc, t0):                                           # punti fitti del percorso e istante [s] di arrivo in ognuno
    s = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(perc, axis=0), axis=1))])
    sf = np.linspace(0, s[-1], max(int(s[-1] / (h / 2)), 2))
    Pf = np.stack([np.interp(sf, s, perc[:, c]) for c in range(3)], axis=1)
    dt = mp["F_i"](((Pf[:-1] + Pf[1:]) / 2)[:, ::-1]) * np.diff(sf) * SEC_PER_U    # F * ds, in secondi
    return Pf, t0 + np.concatenate([[0], np.cumsum(dt)]), sf
def entra(Pf, f):                                                   # indice del primo punto del percorso dentro il foro f (None se non ci passa)
    d = (Pf[:, 0] >= f[0]) & (Pf[:, 0] <= f[1]) & (Pf[:, 1] >= f[2]) & (Pf[:, 1] <= f[3]) & (Pf[:, 2] >= f[4]) & (Pf[:, 2] <= f[5])
    return int(np.argmax(d)) if d.any() else None

def durante_il_ricalcolo(Pf, tf, vf, k, t_ev, pos, v0, t_pronto, mp_nuova):
    # Mentre ricalcola il drone continua sul percorso vecchio (punti k, k+1, ...), che passa ancora dal foro chiuso.
    # Se il prossimo punto e' nel foro chiuso (o nel suo margine) frena e resta fermo fino a t_pronto.
    libero = mp_nuova["peso"](Pf[:, ::-1]) > 0.999
    T, P, V, tl, pl, vl = [], [], [], t_ev, pos, v0
    for i in range(k, len(Pf)):
        if tf[i] >= t_pronto:                                       # il percorso nuovo e' pronto prima di arrivare al punto i
            a = (t_pronto - tl) / (tf[i] - tl)
            return T + [t_pronto], P + [pl + a * (Pf[i] - pl)], V + [vl + a * (vf[i] - vl)], None, False
        if not libero[i]:                                           # il punto i e' nel foro chiuso: il drone si ferma dov'e'
            return T + [min(tl + 1e-3, t_pronto), t_pronto], P + [pl, pl], V + [0.0, 0.0], (tl, t_pronto), False
        T.append(tf[i]); P.append(Pf[i]); V.append(vf[i]); tl, pl, vl = tf[i], Pf[i], vf[i]
    return T, P, V, None, True                                      # arrivato in B mentre ricalcolava

# 3) VOLO: ogni drone vola nella sua copia della stanza; la chiusura j avviene quando QUEL drone e' a ANTICIPO dal foro chiuso[j]
def vola(metodo):
    T, P, V, piani, fermo, ricalcolo, cambi = [0.0], [A.copy()], [None], [], [], [], {}
    pos, t, m = A.copy(), 0.0, 0
    while True:
        Pf, tf, sf = orario(mappe[m], percorso(mappe[m], pos), t)
        vf = V_MAX / mappe[m]["F_i"](Pf[:, ::-1])                 # velocita' [m/s] in ogni punto: v = V_MAX / F
        piani.append((t, Pf, tf))
        if V[-1] is None: V[-1] = vf[0]
        nuova, k = None, None
        for j in range(m + 1, len(mappe)):                          # la prossima chiusura: il primo foro da chiudere che il piano attraversa
            e = entra(Pf, chiuso[j])
            if e is not None:
                nuova, k = j, int(np.searchsorted(sf, sf[e] - ANTICIPO))
                break
        if nuova is None:                                           # nessun'altra chiusura sul percorso: vola fino a B
            T += list(tf[1:]); P += list(Pf[1:]); V += list(vf[1:])
            return dict(T=np.array(T), P=np.array(P), V=np.array(V), piani=piani, fermo=fermo, ricalcolo=ricalcolo, cambi=cambi)
        k = max(k, 1)
        T += list(tf[1:k + 1]); P += list(Pf[1:k + 1]); V += list(vf[1:k + 1])
        t_ev, t_pronto = tf[k], tf[k] + tempi[nuova][metodo]        # il foro si chiude; quando il percorso nuovo e' pronto
        ricalcolo.append((t_ev, t_pronto)); cambi[nuova] = t_ev
        Tr, Pr, Vr, fr, arrivato = durante_il_ricalcolo(Pf, tf, vf, k + 1, t_ev, Pf[k], vf[k], t_pronto, mappe[nuova])
        T += Tr; P += Pr; V += Vr
        if fr: fermo.append(fr)
        if arrivato:
            return dict(T=np.array(T), P=np.array(P), V=np.array(V), piani=piani, fermo=fermo, ricalcolo=ricalcolo, cambi=cambi)
        pos, t, m = np.array(P[-1]), t_pronto, nuova
voli = {met: vola(met) for met in ("Block FIM", "FMM")}
colore = {"Block FIM": BLU, "FMM": ARANCIO}
chiaro = {"Block FIM": "#a9cbf0", "FMM": "#f7c3a8"}                 # sfumatura chiara dello stesso colore (lampeggio)
scena_di = {"Block FIM": "scene", "FMM": "scene2"}
print(f"Livello {LIVELLO} (N = {N}): {len(chiuso)} fori si chiudono durante il volo; tempi di ricalcolo {sorgente}")
for met, v in voli.items():
    ric, fer = [b - a for a, b in v["ricalcolo"]], sum(b - a for a, b in v["fermo"])
    print(f"  {met:9s}: arriva in B dopo {v['T'][-1]:7.2f} s; ricalcola per {sum(ric):6.3f} s "
          f"({' + '.join(f'{a * 1000:.0f} ms' for a in ric)}), di cui {fer:6.3f} s fermo davanti a un foro chiuso")
rit = voli["FMM"]["T"][-1] - voli["Block FIM"]["T"][-1]
print(f"  il drone del FMM arriva {rit:.2f} s dopo quello del Block FIM")

# 4) GEOMETRIA: i muri con tutti i fori aperti (mappa iniziale), i fori che si chiudono compaiono in rosso
amb0 = mappe[0]["amb"]
interni = [b for b in amb0["SCATOLA"] if b[0] > -0.99 and b[1] < 0.99]
fori_di = lambda b: [f for f in amb0["FORO"] if b[0] - 1e-9 <= f[0] and f[1] <= b[1] + 1e-9]
def superficie(b, fori):                                            # facce esterne del muro b con i suoi fori: X, Y, Z, I, J, K
    tagli = [sorted({b[2 * a], b[2 * a + 1]} | {min(max(f[2 * a + s], b[2 * a]), b[2 * a + 1]) for f in fori for s in (0, 1)}) for a in range(3)]
    n = [len(t) - 1 for t in tagli]
    pieno = np.zeros(n, bool)
    for ix in range(n[0]):
        for iy in range(n[1]):
            for iz in range(n[2]):
                c = [(tagli[a][i] + tagli[a][i + 1]) / 2 for a, i in enumerate((ix, iy, iz))]
                pieno[ix, iy, iz] = not any(f[0] <= c[0] <= f[1] and f[2] <= c[1] <= f[3] and f[4] <= c[2] <= f[5] for f in fori)
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for ix, iy, iz in zip(*np.nonzero(pieno)):
        for asse in range(3):
            for verso in (0, 1):
                vic = [ix, iy, iz]; vic[asse] += 1 if verso else -1
                if 0 <= vic[asse] < n[asse] and pieno[tuple(vic)]: continue      # faccia interna: non si disegna
                lo = [tagli[a][i] for a, i in enumerate((ix, iy, iz))]; hi = [tagli[a][i + 1] for a, i in enumerate((ix, iy, iz))]
                q = lo[asse] if verso == 0 else hi[asse]
                u_, w_ = [a for a in range(3) if a != asse]
                o = len(X)
                for du, dw in [(0, 0), (1, 0), (1, 1), (0, 1)]:
                    v = [0.0] * 3; v[asse] = q; v[u_] = (lo[u_], hi[u_])[du]; v[w_] = (lo[w_], hi[w_])[dw]
                    X.append(v[0]); Y.append(v[1]); Z.append(v[2])
                I += [o, o]; J += [o + 1, o + 2]; K += [o + 2, o + 3]
    return X, Y, Z, I, J, K
X, Y, Z, I, J, K = [], [], [], [], [], []
for b in interni:
    Xb, Yb, Zb, Ib, Jb, Kb = superficie(b, fori_di(b)); o = len(X)
    X += Xb; Y += Yb; Z += Zb; I += [o + i for i in Ib]; J += [o + j for j in Jb]; K += [o + k for k in Kb]
muri = go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color="#9a9a9a", opacity=0.35, name=f"{len(interni)} muri con i fori", flatshading=True, hoverinfo="name")
spigoli = []
for a in (-1, 1):
    for b in (-1, 1):
        spigoli += [[(-1, a, b), (1, a, b)], [(a, -1, b), (a, 1, b)], [(a, b, -1), (a, b, 1)]]
stanza = go.Scatter3d(x=sum([[p[0][0], p[1][0], None] for p in spigoli], []), y=sum([[p[0][1], p[1][1], None] for p in spigoli], []),
                      z=sum([[p[0][2], p[1][2], None] for p in spigoli], []), mode="lines", line=dict(color="gray", width=2),
                      name="stanza (20 m)", hoverinfo="skip")
ab = go.Scatter3d(x=[A[0], B[0]], y=[A[1], B[1]], z=[A[2], B[2]], mode="markers+text", text=["  A (partenza)", "  B (arrivo)"],
                  marker=dict(size=8, color=["green", "black"]), textfont=dict(size=14), name="A e B")
def scatola(f, scena):                                              # il foro f chiuso: una scatola rossa
    X = [f[0], f[1], f[1], f[0], f[0], f[1], f[1], f[0]]; Y = [f[2], f[2], f[3], f[3]] * 2; Z = [f[4]] * 4 + [f[5]] * 4
    I, J, K = zip(*[(0, 1, 2), (0, 2, 3), (4, 5, 6), (4, 6, 7), (0, 1, 5), (0, 5, 4), (1, 2, 6), (1, 6, 5), (2, 3, 7), (2, 7, 6), (3, 0, 4), (3, 4, 7)])
    return go.Mesh3d(x=X, y=Y, z=Z, i=list(I), j=list(J), k=list(K), color=ROSSO, opacity=0.85, scene=scena, showlegend=False, name="foro chiuso", flatshading=True)
vuoto = lambda scena: go.Mesh3d(x=[], y=[], z=[], i=[], j=[], k=[], scene=scena, showlegend=False)

# 5) ANIMAZIONE affiancata: a sinistra il drone del Block FIM, a destra quello del FMM (stessi istanti)
def posizione(met, t):
    v = voli[met]; T, P, V = v["T"], v["P"], v["V"]
    if t >= T[-1]: return P[-1], 0.0
    k = np.searchsorted(T, t, side="right"); a = (t - T[k - 1]) / max(T[k] - T[k - 1], 1e-12)
    return P[k - 1] + a * (P[k] - P[k - 1]), V[k - 1] + a * (V[k] - V[k - 1])
def drone(met, t):                                                  # scia colorata con la velocita', percorso previsto e drone
    v = voli[met]; T, P, V = v["T"], v["P"], v["V"]
    pos, vel = posizione(met, t)
    fermo = any(a <= t < b for a, b in v["fermo"])
    ricalcola = any(a <= t < b for a, b in v["ricalcolo"])          # mentre ricalcola il pallino lampeggia
    piano = [p for p in v["piani"] if p[0] <= t][-1]
    resto = np.vstack([pos[None], piano[1][piano[2] > t]]) if not fermo and t < T[-1] else np.empty((0, 3))
    passo = max(1, int((T <= t).sum() / 400))                       # al massimo ~400 punti nella scia
    scia, vs = np.vstack([P[T <= t][::passo], pos[None]]), np.append(V[T <= t][::passo], vel)
    sc, c = scena_di[met], colore[met]
    return [go.Scatter3d(x=scia[:, 0], y=scia[:, 1], z=scia[:, 2], mode="lines", scene=sc, showlegend=False, name=f"scia {met}",
                         line=dict(color=vs, colorscale="Turbo", cmin=0.0, cmax=V_MAX, width=12,
                                   colorbar=dict(title="velocita'<br>[m/s]", x=0.47, len=0.6) if met == "Block FIM" else None,
                                   showscale=(met == "Block FIM"))),
            go.Scatter3d(x=resto[:, 0], y=resto[:, 1], z=resto[:, 2], mode="lines", scene=sc, showlegend=False,
                         line=dict(color="black", width=2, dash="dash"), name=f"percorso previsto {met}"),
            go.Scatter3d(x=[pos[0]], y=[pos[1]], z=[pos[2]], mode="markers", scene=sc, showlegend=False, name=f"drone {met}",
                         marker=dict(size=7, color=chiaro[met] if ricalcola and round(t / DT_FRAME) % 2 else c, line=dict(color=c, width=5)))]
def stato(met, t):
    v = voli[met]
    if t >= v["T"][-1]: return "arrivato"
    if any(a <= t < b for a, b in v["fermo"]): return "FERMO davanti al foro chiuso, ricalcola"
    if any(a <= t < b for a, b in v["ricalcolo"]): return f"ricalcola, vola sul percorso vecchio a {posizione(met, t)[1]:.1f} m/s"
    return f"in volo a {posizione(met, t)[1]:.1f} m/s"
def dati(t):
    out = []
    for met in ("Block FIM", "FMM"):
        sc = scena_di[met]
        for j in chiuso:                                            # il foro j compare (rosso) quando si chiude per questo drone
            out.append(scatola(chiuso[j], sc) if voli[met]["cambi"].get(j, np.inf) <= t else vuoto(sc))
        out += drone(met, t)
    return out
def titolo(t):
    return (f"Livello {LIVELLO}, t = {t:5.1f} s<br><span style='color:{BLU}'>Block FIM: {stato('Block FIM', t)}</span>  |  "
            f"<span style='color:{ARANCIO}'>FMM: {stato('FMM', t)}</span>")
t_fine = max(v["T"][-1] for v in voli.values())
DT_FRAME = max(0.3, t_fine / 400)                                   # al massimo ~400 fotogrammi (file HTML leggero anche per i voli lunghi)
istanti = np.append(np.arange(0.0, t_fine, DT_FRAME), t_fine)
fisse = []
for sc in ("scene", "scene2"):                                      # stanza, muri, A e B: in tutte e due le viste
    for tr in (stanza, muri, ab):
        tr = go.Figure(tr).data[0]; tr.update(scene=sc, showlegend=(sc == "scene")); fisse.append(tr)
mobili = list(range(len(fisse), len(fisse) + len(dati(0.0))))
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.02,
                    subplot_titles=(f"Block FIM (GPU): arriva a {voli['Block FIM']['T'][-1]:.1f} s",
                                    f"FMM (CPU): arriva a {voli['FMM']['T'][-1]:.1f} s"))
for tr in fisse + dati(0.0): fig.add_trace(tr)
fig.frames = [go.Frame(data=dati(t), traces=mobili, name=f"{t:.2f}", layout=dict(title=dict(text=titolo(t)))) for t in istanti]
ASSI = dict(range=[-1.0, 1.0], autorange=False, showticklabels=False, title="")
SCENA = dict(xaxis=ASSI, yaxis=ASSI, zaxis=ASSI, aspectmode="cube", camera=dict(eye=dict(x=0.3, y=-1.9, z=1.1)))
fig.update_layout(title=dict(text=titolo(0.0), x=0.5, font=dict(size=14)), scene=SCENA, scene2=SCENA, height=850,
                  margin=dict(l=0, r=0, t=90, b=230), legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.02),
                  updatemenus=[dict(type="buttons", direction="left", x=0.08, y=0.0, xanchor="right", yanchor="top", pad=dict(r=10, t=60), buttons=[
                      dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=MS_FRAME, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                      dict(label="Pausa", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
                  sliders=[dict(x=0.1, len=0.88, y=0.0, yanchor="top", currentvalue=dict(prefix="tempo di volo t = ", suffix=" s"), pad=dict(t=40),
                                steps=[dict(label="", method="animate", args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                                       for t in istanti])])
fig.add_annotation(xref="paper", yref="paper", x=0.5, y=-0.3, xanchor="center", yanchor="top", showarrow=False, font=dict(size=12),
                   text=(f"Tempi di ricalcolo {sorgente}. 1 unita' = {METRI_PER_UNITA:.0f} m, V_MAX = {V_MAX:.0f} m/s: tempo di volo = u x {SEC_PER_U:.0f} s. "
                         f"Play: 1 fotogramma = {DT_FRAME:.2f} s di volo ogni {MS_FRAME} ms ({DT_FRAME * 1000 / MS_FRAME:.0f} volte piu' veloce del vero)."))
fig.write_html(f"gara_ripianificazione_livello{LIVELLO}.html", include_plotlyjs="cdn")    # anche come file da scaricare
fig.show()

# 6) VELOCITA' NEL TEMPO dei due droni (fasce = ricalcola; linee = un foro si chiude)
g = go.Figure()
for met, v in voli.items():
    g.add_trace(go.Scatter(x=v["T"], y=v["V"], mode="lines", line=dict(color=colore[met], width=2.5), name=f"{met}: arriva a {v['T'][-1]:.1f} s"))
    for a, b in v["ricalcolo"]:
        g.add_vrect(x0=a, x1=max(b, a + 0.05), fillcolor=colore[met], opacity=0.2, line_width=0)
    for j, t_ev in v["cambi"].items():
        g.add_vline(x=t_ev, line=dict(color=colore[met], dash="dot"))
g.add_hline(y=V_MAX, line=dict(color="gray", dash="dash"))
g.update_layout(template="plotly_white", height=380, xaxis_title="tempo di volo [s]", yaxis_title="velocita' [m/s]",
                legend=dict(orientation="h", y=-0.25), title=f"Livello {LIVELLO}: il drone del FMM arriva {rit:.1f} s dopo")
g.show()
